# Dispersão das temperaturas: SONDA-MD × NASA POWER

Este notebook reúne os horários coincidentes das bases tratadas da SONDA-MD e da NASA POWER e gera um gráfico de dispersão com:

- temperatura da SONDA-MD no eixo horizontal;
- temperatura da NASA POWER no eixo vertical;
- um ponto para cada horário coincidente;
- linha de referência `y = x`; e
- correlação de Pearson.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import pearsonr


def localizar_dados_tratados():
    # Permite executar o notebook a partir do projeto ou da pasta notebooks.
    for base in (Path.cwd(), *Path.cwd().parents):
        candidatos = (
            base / "2026" / "dados_tratados",
            base.parent / "2026" / "dados_tratados",
        )
        for candidato in candidatos:
            if (candidato / "sonda_md_horario_padronizado.csv").exists() and (
                candidato / "nasa_power_horario_padronizado.csv"
            ).exists():
                return candidato.resolve()
    raise FileNotFoundError(
        "Não foi possível localizar a pasta 2026/dados_tratados."
    )


PASTA_DADOS = localizar_dados_tratados()
ARQUIVO_SONDA = PASTA_DADOS / "sonda_md_horario_padronizado.csv"
ARQUIVO_NASA = PASTA_DADOS / "nasa_power_horario_padronizado.csv"

sonda = pd.read_csv(ARQUIVO_SONDA, parse_dates=["data_hora"])
nasa = pd.read_csv(ARQUIVO_NASA, parse_dates=["data_hora"])

dados = (
    sonda[["data_hora", "temperatura"]]
    .merge(
        nasa[["data_hora", "temperatura"]],
        on="data_hora",
        how="inner",
        suffixes=("_sonda_md", "_nasa_power"),
    )
    .dropna(subset=["temperatura_sonda_md", "temperatura_nasa_power"])
    .sort_values("data_hora")
    .reset_index(drop=True)
)

if len(dados) != 17_568:
    raise ValueError(f"Esperados 17.568 pares válidos; encontrados {len(dados):,}.")

r, p_valor = pearsonr(
    dados["temperatura_sonda_md"],
    dados["temperatura_nasa_power"],
)

n_formatado = f"{len(dados):,}".replace(",", ".")
r_formatado = f"{r:.3f}".replace(".", ",")
print(f"Pares horários coincidentes: {n_formatado}")
print(f"Período: {dados['data_hora'].min()} a {dados['data_hora'].max()}")
print(f"Correlação de Pearson: r = {r_formatado}")


In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 7), dpi=150)

ax.scatter(
    dados["temperatura_sonda_md"],
    dados["temperatura_nasa_power"],
    s=9,
    alpha=0.28,
    color="#2166ac",
    edgecolors="none",
    rasterized=True,
)

limite_min = min(
    dados["temperatura_sonda_md"].min(),
    dados["temperatura_nasa_power"].min(),
)
limite_max = max(
    dados["temperatura_sonda_md"].max(),
    dados["temperatura_nasa_power"].max(),
)
margem = 0.03 * (limite_max - limite_min)
limites = [limite_min - margem, limite_max + margem]
ax.plot(limites, limites, color="#333333", linewidth=1.4, label="y = x")

ax.set_xlim(limites)
ax.set_ylim(limites)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Temperatura da SONDA-MD (°C)")
ax.set_ylabel("Temperatura da NASA POWER (°C)")
ax.set_title("Temperaturas horárias coincidentes: SONDA-MD × NASA POWER")
ax.text(
    0.04,
    0.96,
    f"n = {n_formatado}\nPearson: r = {r_formatado}",
    transform=ax.transAxes,
    ha="left",
    va="top",
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.9, edgecolor="0.7"),
)
ax.grid(True, linestyle=":", alpha=0.35)
ax.legend(loc="lower right")
fig.tight_layout()

PASTA_SAIDA = PASTA_DADOS.parent / "resultados" / "figuras" / "comparacao_temperaturas"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
ARQUIVO_FIGURA = PASTA_SAIDA / "dispersao_temperatura_sonda_md_nasa_power.png"
fig.savefig(ARQUIVO_FIGURA, dpi=300, bbox_inches="tight")
print(f"Figura salva em: {ARQUIVO_FIGURA}")
plt.show()
